In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

columnas_a_probar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_tfidf = []


for col in columnas_a_probar:
    X = df[col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    vectorizador_tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = vectorizador_tfidf.fit_transform(X_train)
    X_test_tfidf = vectorizador_tfidf.transform(X_test)
    
    modelo_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_tfidf, y_train)
    score_lr = f1_score(y_test, modelo_lr.predict(X_test_tfidf), average='macro')
    
    modelo_nb = MultinomialNB()
    modelo_nb.fit(X_train_tfidf, y_train)
    score_nb = f1_score(y_test, modelo_nb.predict(X_test_tfidf), average='macro')
    
    modelo_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
    modelo_svm.fit(X_train_tfidf, y_train)
    score_svm = f1_score(y_test, modelo_svm.predict(X_test_tfidf), average='macro')
    
    resultados_tfidf.append({
        'Preprocesamiento': col, 
        'Macro-F1 (Reg. Logística)': score_lr,
        'Macro-F1 (Naive Bayes)': score_nb,
        'Macro-F1 (SVM)': score_svm
    })

df_resultados_tfidf = pd.DataFrame(resultados_tfidf).sort_values(by='Macro-F1 (Reg. Logística)', ascending=False)

print(df_resultados_tfidf.to_string(index=False))

Preprocesamiento  Macro-F1 (Reg. Logística)  Macro-F1 (Naive Bayes)  Macro-F1 (SVM)
            text                   0.621152                0.461273        0.684853
  text_lema_nltk                   0.579998                0.463813        0.603238
     text_basico                   0.578737                0.465567        0.597422
  text_stem_nltk                   0.578697                0.456440        0.608222
       text_lema                   0.578437                0.463362        0.599237
        text_pos                   0.576052                0.452919        0.587571


Realizamos lo mismo que anteriormente pero esta vez realizamos un pesado binario y comparamos los resultados para obtener:

-Cual es el mejor preprocesamieneto para este caso (o si no es necesario)

-Cual es la mejor representacion

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

y = df['t']

resultados = []

for col in columnas_a_probar:
    X = df[col]
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    vectorizador_bin = CountVectorizer(max_features=5000, binary=True)
    X_train_bin = vectorizador_bin.fit_transform(X_train)
    X_test_bin = vectorizador_bin.transform(X_test)
    
    modelo_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_bin, y_train)
    score_lr = f1_score(y_test, modelo_lr.predict(X_test_bin), average='macro')
    
    modelo_nb = MultinomialNB()
    modelo_nb.fit(X_train_bin, y_train)
    score_nb = f1_score(y_test, modelo_nb.predict(X_test_bin), average='macro')
    
    modelo_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
    modelo_svm.fit(X_train_bin, y_train)
    score_svm = f1_score(y_test, modelo_svm.predict(X_test_bin), average='macro')
    
    resultados.append({
        'Preprocesamiento': col, 
        'Macro-F1 (Reg. Logística)': score_lr,
        'Macro-F1 (Naive Bayes)': score_nb,
        'Macro-F1 (SVM)': score_svm
    })

df_resultados = pd.DataFrame(resultados).sort_values(by='Macro-F1 (Reg. Logística)', ascending=False)

print(df_resultados.to_string(index=False))

Preprocesamiento  Macro-F1 (Reg. Logística)  Macro-F1 (Naive Bayes)  Macro-F1 (SVM)
            text                   0.704029                0.517240        0.662412
  text_lema_nltk                   0.569591                0.501426        0.520401
  text_stem_nltk                   0.568654                0.508922        0.525468
       text_lema                   0.568181                0.505213        0.537740
     text_basico                   0.553169                0.501394        0.526847
        text_pos                   0.536594                0.515326        0.497038


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer # Usamos la versión estándar
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

columnas_a_probar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_frecuencia = []

for col in columnas_a_probar:
    X = df[col].astype(str)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    # Aplicamos Frecuencia Absoluta (Conteo estándar de palabras)
    vectorizador_frecuencia = CountVectorizer(max_features=5000)
    X_train_freq = vectorizador_frecuencia.fit_transform(X_train)
    X_test_freq = vectorizador_frecuencia.transform(X_test)
    
    # 1. Regresión Logística
    modelo_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_freq, y_train)
    score_lr = f1_score(y_test, modelo_lr.predict(X_test_freq), average='macro')
    
    # 2. Naive Bayes
    modelo_nb = MultinomialNB()
    modelo_nb.fit(X_train_freq, y_train)
    score_nb = f1_score(y_test, modelo_nb.predict(X_test_freq), average='macro')
    
    # 3. SVM (Linear Support Vector Classification)
    modelo_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
    modelo_svm.fit(X_train_freq, y_train)
    score_svm = f1_score(y_test, modelo_svm.predict(X_test_freq), average='macro')
    
    resultados_frecuencia.append({
        'Preprocesamiento': col, 
        'Macro-F1 (Reg. Logística)': score_lr,
        'Macro-F1 (Naive Bayes)': score_nb,
        'Macro-F1 (SVM)': score_svm
    })

df_resultados_freq = pd.DataFrame(resultados_frecuencia).sort_values(by='Macro-F1 (Reg. Logística)', ascending=False)

print("\n📊 RESULTADOS CON FRECUENCIA ABSOLUTA (Bag of Words) 📊\n")
print(df_resultados_freq.to_string(index=False))

c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users


📊 RESULTADOS CON FRECUENCIA ABSOLUTA (Bag of Words) 📊

Preprocesamiento  Macro-F1 (Reg. Logística)  Macro-F1 (Naive Bayes)  Macro-F1 (SVM)
            text                   0.674111                0.508917        0.620780
  text_stem_nltk                   0.580612                0.496396        0.535791
        text_pos                   0.573829                0.512701        0.546887
  text_lema_nltk                   0.570948                0.502966        0.545655
       text_lema                   0.570787                0.502927        0.537148
     text_basico                   0.560919                0.500346        0.524851


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
